# Multi-Source Excel Consolidation Workflow — Public Demo (2024)

This notebook is a **sanitized portfolio version** of a real operational data-consolidation workflow.

The original project combined a master Excel tracking file with several specialist workbooks.  
To protect confidential information, this notebook uses **synthetic records, generic field names, and generic source labels**.

The workflow demonstrates:

- loading a master dataset and multiple update sources
- validating unique record identifiers
- normalizing values before comparison
- comparing records cell by cell
- separating unique updates from conflicts
- applying only validated updates
- keeping a change log for traceability
- validating the consolidated result before export

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Create a synthetic master dataset

In the real project, the master file was the official tracking workbook.  
Here we create a small synthetic dataset so the notebook can run without any private files.

In [ ]:
master = pd.DataFrame({
    "RECORD_ID": [f"REC-{i:03d}" for i in range(1, 9)],
    "REGION": ["NORTH", "SOUTH", "CENTRAL", "NORTH", "SOUTH", "CENTRAL", "NORTH", "SOUTH"],
    "STATUS_A": ["DONE", "PENDING", "DONE", "DONE", "PENDING", "DONE", "PENDING", "DONE"],
    "STATUS_B": ["PENDING", "PENDING", "DONE", "PENDING", "PENDING", "DONE", "PENDING", "DONE"],
    "OWNER": ["ANALYST_A", "ANALYST_A", "ANALYST_B", "ANALYST_B", "ANALYST_A", "ANALYST_B", "ANALYST_A", "ANALYST_B"]
})

master

## 2. Create synthetic specialist updates

Each source represents a workbook completed by a different specialist.

In [ ]:
specialist_1 = master.copy()
specialist_1.loc[specialist_1["RECORD_ID"] == "REC-002", "STATUS_A"] = "DONE"
specialist_1.loc[specialist_1["RECORD_ID"] == "REC-004", "STATUS_B"] = "DONE"

specialist_2 = master.copy()
specialist_2.loc[specialist_2["RECORD_ID"] == "REC-005", "STATUS_A"] = "DONE"
specialist_2.loc[specialist_2["RECORD_ID"] == "REC-007", "OWNER"] = "ANALYST_C"

sources = {
    "SOURCE_1": specialist_1,
    "SOURCE_2": specialist_2
}

## 3. Validate the record key

A reliable key is required before using record-level comparisons.

In [ ]:
print("Rows:", len(master))
print("Unique IDs:", master["RECORD_ID"].nunique())
print("Duplicate IDs:", master["RECORD_ID"].duplicated().sum())

assert master["RECORD_ID"].duplicated().sum() == 0

## 4. Normalize values before comparison

Excel data often contains extra spaces, blank strings, line breaks, or missing values.

In [ ]:
def normalize_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, str):
        value = " ".join(value.replace("\n", " ").replace("\r", " ").split()).strip()
        return value if value else None

    return value

## 5. Compare the master dataset with every update source

The comparison is performed by `RECORD_ID` and column.

In [ ]:
master_idx = master.set_index("RECORD_ID")
source_idx = {
    name: df.set_index("RECORD_ID")
    for name, df in sources.items()
}

differences = []

for source_name, df_source in source_idx.items():
    for record_id in master_idx.index:
        for column in master_idx.columns:
            base_value = normalize_value(master_idx.at[record_id, column])
            source_value = normalize_value(df_source.at[record_id, column])

            if base_value != source_value:
                differences.append({
                    "SOURCE": source_name,
                    "RECORD_ID": record_id,
                    "COLUMN": column,
                    "BASE_VALUE": base_value,
                    "SOURCE_VALUE": source_value
                })

differences_df = pd.DataFrame(differences)
differences_df

## 6. Classify unique updates and conflicts

A **unique update** occurs when one new value is proposed for a cell.  
A **conflict** occurs when different sources propose different values for the same cell.

In [ ]:
grouped = (
    differences_df
    .groupby(["RECORD_ID", "COLUMN"])["SOURCE_VALUE"]
    .agg(lambda s: list(pd.unique(s)))
    .reset_index(name="PROPOSED_VALUES")
)

grouped["VALUE_COUNT"] = grouped["PROPOSED_VALUES"].apply(len)

unique_updates = grouped[grouped["VALUE_COUNT"] == 1].copy()
conflicts = grouped[grouped["VALUE_COUNT"] > 1].copy()

print("Unique updates:", len(unique_updates))
print("Conflicts:", len(conflicts))

## 7. Apply validated unique updates

Only cells with one unambiguous proposed value are automatically updated.

In [ ]:
consolidated = master_idx.copy()
traceability = []

for _, row in unique_updates.iterrows():
    record_id = row["RECORD_ID"]
    column = row["COLUMN"]
    new_value = row["PROPOSED_VALUES"][0]
    old_value = consolidated.at[record_id, column]

    consolidated.at[record_id, column] = new_value

    traceability.append({
        "RECORD_ID": record_id,
        "COLUMN": column,
        "OLD_VALUE": old_value,
        "NEW_VALUE": new_value,
        "DECISION": "UNIQUE_UPDATE"
    })

consolidated = consolidated.reset_index()
traceability_df = pd.DataFrame(traceability)

consolidated

## 8. Final validation

A consolidation workflow should not be considered complete only because the code ran successfully.
Basic quality checks are performed before export.

In [ ]:
print("Final rows:", len(consolidated))
print("Unique IDs:", consolidated["RECORD_ID"].nunique())
print("Duplicate IDs:", consolidated["RECORD_ID"].duplicated().sum())
print("Applied changes:", len(traceability_df))

assert len(consolidated) == consolidated["RECORD_ID"].nunique()
assert consolidated["RECORD_ID"].duplicated().sum() == 0

## 9. Export example

The public demo writes only synthetic data.

In [ ]:
output_dir = Path("demo_output")
output_dir.mkdir(exist_ok=True)

consolidated.to_excel(
    output_dir / "consolidated_demo_2024.xlsx",
    index=False
)

traceability_df.to_excel(
    output_dir / "traceability_demo_2024.xlsx",
    index=False
)

print("Demo files exported.")

## Key Takeaway

The central idea is not simply to append Excel files.  
The workflow treats the master workbook as the reference dataset, detects cell-level changes, separates safe updates from conflicts, and validates the final result before reporting.